# Natural Language Processing Lab
## Experiment 1: Working with Text Input and Python Data Structures
**Domain Application:** Terms and Conditions Summarizer (Amazon & Alibaba Agreements)

### Program 1: Interactive Clause Input & Basic String Manipulation
Accept raw legal text from user input and perform string-level statistical inspection.

In [ ]:
# Program 1: String operations on a sample Terms & Conditions clause
sample_clause = "Amazon reserves the right to refuse service, terminate accounts, or cancel orders in its sole discretion."

print("--- Raw Legal Clause ---")
print(sample_clause)
print("\n--- String Analysis ---")
print("Total Characters (with spaces):", len(sample_clause))
print("Total Characters (without spaces):", len(sample_clause.replace(" ", "")))
print("Total Words:", len(sample_clause.split()))
print("Uppercase Conversion:", sample_clause.upper())
print("Lowercase Conversion:", sample_clause.lower())
print("Contains keyword 'terminate':", "terminate" in sample_clause.lower())

### Program 2: Safe File Reading with UTF-8 Encoding
Read unstructured text files (`amazon.txt` & `alibaba.txt`) using robust exception handling.

In [ ]:
# Program 2: Safe file reading function for T&C documents
def read_tc_document(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()
            print(f"[SUCCESS] Successfully loaded '{file_path}' ({len(content)} characters)")
            return content
    except FileNotFoundError:
        print(f"[ERROR] File not found: {file_path}")
        return None
    except UnicodeDecodeError:
        print(f"[ERROR] Unable to decode file with UTF-8: {file_path}")
        return None

# Read Amazon & Alibaba agreement files
amazon_text = read_tc_document("data/amazon.txt")
alibaba_text = read_tc_document("data/alibaba.txt")

### Program 3: Line-by-Line Ingestion, Tuples, and Unique Vocabulary Sets
Parse legal documents into clauses (`list`), create `(clause_index, word_count)` metadata pairs (`tuple`), and compute unique vocabulary sets (`set`).

In [ ]:
# Program 3: Line-by-line reading into list of clauses
def extract_clauses(file_path):
    clauses = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            cleaned = line.strip()
            if cleaned and not cleaned.isupper():  # Filter blank lines and standalone headers
                clauses.append(cleaned)
    return clauses

amazon_clauses = extract_clauses("data/amazon.txt")
alibaba_clauses = extract_clauses("data/alibaba.txt")

print(f"Amazon Clauses Extracted: {len(amazon_clauses)}")
print(f"Alibaba Clauses Extracted: {len(alibaba_clauses)}")

# 1. Store as Tuples: (Clause_Index, Word_Count, First_50_Chars)
amazon_clause_tuples = [(idx + 1, len(c.split()), c[:50] + "...") for idx, c in enumerate(amazon_clauses)]
print("\n--- Sample Amazon Clause Tuples (First 3) ---")
for t in amazon_clause_tuples[:3]:
    print(t)

# 2. Store as Sets: Unique Vocabulary Extraction
amazon_words = set(w.lower().strip(".,()[]\"") for c in amazon_clauses for w in c.split())
alibaba_words = set(w.lower().strip(".,()[]\"") for c in alibaba_clauses for w in c.split())

print(f"\nAmazon Unique Vocabulary Size: {len(amazon_words)}")
print(f"Alibaba Unique Vocabulary Size: {len(alibaba_words)}")
common_legal_terms = amazon_words.intersection(alibaba_words)
print(f"Shared Legal Terms Count: {len(common_legal_terms)}")
print("Sample Shared Terms:", list(common_legal_terms)[:10])

### Program 4: Structured CSV Ingestion & Dataset Inspection with Pandas
Load `tc_clauses.csv` and inspect dataset dimensions, data types, null values, and category frequencies.

In [ ]:
# Program 4: Tabular Data Processing with Pandas
import pandas as pd

# Load structured T&C dataset
df_clauses = pd.read_csv("data/tc_clauses.csv")

print("--- Dataset Shape (Rows, Columns) ---")
print(df_clauses.shape)

print("\n--- First 5 Rows ---")
display(df_clauses.head())

print("\n--- Missing Values Check ---")
print(df_clauses.isnull().sum())

print("\n--- Risk Level Distribution ---")
print(df_clauses["risk_level"].value_counts())

print("\n--- Category Distribution across Companies ---")
print(pd.crosstab(df_clauses["company"], df_clauses["risk_level"]))

# Feature engineering: Add word count & character count columns
df_clauses["word_count"] = df_clauses["clause_text"].apply(lambda x: len(str(x).split()))
df_clauses["char_count"] = df_clauses["clause_text"].apply(lambda x: len(str(x)))

print("\n--- DataFrame with NLP Feature Columns ---")
display(df_clauses[["clause_id", "company", "category", "risk_level", "word_count", "char_count"]].head())

### Program 5: Hierarchical JSON Ingestion & Dictionary Mapping
Parse nested `tc_policies.json` and build lookup dictionaries mapping company sections to their policy clauses.

In [ ]:
# Program 5: Parsing Nested JSON and Dictionary Operations
import json

with open("data/tc_policies.json", "r", encoding="utf-8") as file:
    policies_data = json.load(file)

# Build lookup dictionary: {Company -> {Section_Name -> [Clauses]}}
policy_lookup = {}

for policy in policies_data["policies"]:
    company = policy["company"]
    policy_lookup[company] = {}
    for sec in policy["sections"]:
        sec_name = sec["section_name"]
        policy_lookup[company][sec_name] = sec["clauses"]

# Inspect dictionary structure
print("Companies Indexed:", list(policy_lookup.keys()))
print("\nAmazon Sections:", list(policy_lookup["Amazon"].keys()))
print("Alibaba Sections:", list(policy_lookup["Alibaba"].keys()))

print("\n--- Specific Lookup: Alibaba Limitation of Liability Clauses ---")
for idx, clause in enumerate(policy_lookup["Alibaba"]["Limitation of Liability"], start=1):
    print(f"{idx}. {clause}")

### Extension Activity: Statistical Comparison of Amazon vs Alibaba Clauses
Aggregate and compare average word count and clause length metrics by company and risk level.

In [ ]:
# Extension Activity: Comprehensive Summary DataFrame
summary_stats = df_clauses.groupby(["company", "risk_level"]).agg(
    total_clauses=("clause_id", "count"),
    avg_words=("word_count", "mean"),
    avg_chars=("char_count", "mean"),
    max_words=("word_count", "max"),
    min_words=("word_count", "min")
).reset_index()

summary_stats["avg_words"] = summary_stats["avg_words"].round(2)
summary_stats["avg_chars"] = summary_stats["avg_chars"].round(2)

print("--- Summary Statistics by Company and Risk Level ---")
display(summary_stats)